# Desafio Lighthouse 2026.2 — LH Nautical

Notebook com o raciocínio das questões Q1 a Q5: o que cada questão pede, o caminho escolhido, o ponto de atenção principal, a consulta executada, o resultado e a interpretação.

A entrega formal de cada questão está em `../Submissão/Qx/`. A Q2 e a Q3 alteram o banco (geram o schema e carregam os dados), por isso rodam via terminal antes deste notebook abrir — aqui elas aparecem como evidência consultada no banco já processado, não como repetição do script. Já a Q4 e a Q5 reaproveitam a consulta SQL diretamente nas células, o que é adequado para expor o raciocínio passo a passo.

**Ordem de execução:** este notebook espera que o schema (Q2) e a carga (Q3) já tenham rodado via terminal — ver `../README.md`.

## Configuração

In [1]:
import pandas as pd
import psycopg2

# Credenciais vêm das variáveis de ambiente padrão do PostgreSQL
# (PGHOST, PGPORT, PGDATABASE, PGUSER, PGPASSWORD) — nunca no código.
pg = psycopg2.connect()


def consultar(sql_texto):
    # Cursor puro do psycopg2 — evita o aviso do pandas sobre conexão
    # não ser SQLAlchemy, sem precisar adicionar essa dependência.
    with pg.cursor() as cursor:
        cursor.execute(sql_texto)
        colunas = [descricao[0] for descricao in cursor.description]
        return pd.DataFrame(cursor.fetchall(), columns=colunas)

## Q1 — EDA da tabela `orders`

### A. Objetivo

O Sr. Almir quer saber se pode confiar nos dados de `orders` pra tomar decisão. A questão pede uma EDA simples — linhas, colunas, intervalo de datas, estatísticas de `total` — e um diagnóstico curto sobre qualidade e prontidão dos dados.

### B. Caminho adotado

Inspeção inicial em pandas, direto no CSV bruto. A consulta SQL pedida pela Q1.1 roda no mesmo PostgreSQL que as Q2/Q3 constroem — não precisa de nenhuma outra ferramenta.

### C. Ponto de atenção

`COUNT(*)` conta linha, não coluna. Contar colunas certo em SQL usa `information_schema.columns`, filtrando pelo schema atual.

### D. Consulta ou código — inspeção inicial em pandas

In [2]:
orders_csv = pd.read_csv("../data/raw/1-lh_nautical_csv/orders.csv")
print("linhas, colunas:", orders_csv.shape)
orders_csv.head()

linhas, colunas: (48998, 13)


,id,order_number,channel,customer_id,salesperson_id,location_id,status,subtotal,discount_amount,total,placed_at,created_at,updated_at
0,1,SO-000001,ecommerce,1136,NaN,1,paid,323.34,35.57,287.77,2022-09-06 05:37:37,2022-09-06 05:37:37,2022-09-06 05:37:37
1,2,SO-000002,ecommerce,618,9.0,4,paid,53199.05,0.00,53199.05,2023-02-03 04:36:21,2023-02-03 04:36:21,2023-02-03 04:36:21
2,3,SO-000003,pos,227,10.0,4,confirmed,17157.39,0.00,17157.39,2024-12-30 07:15:17,2024-12-30 07:15:17,2024-12-30 07:15:17
3,4,SO-000004,ecommerce,1123,NaN,3,paid,11095.62,665.74,10429.88,2024-03-12 20:39:36,2024-03-12 20:39:36,2024-03-12 20:39:36
4,5,SO-000005,ecommerce,426,9.0,2,confirmed,32842.03,0.00,32842.03,2020-03-27 13:52:12,2020-03-27 13:52:12,2020-03-27 13:52:12


In [3]:
print("nulos por coluna:")
print(orders_csv.isnull().sum())
print("\nid duplicado:", orders_csv["id"].duplicated().sum())
print("order_number duplicado:", orders_csv["order_number"].duplicated().sum())

nulos por coluna:
id                     0
order_number           0
channel                0
customer_id            0
salesperson_id     24131
location_id            0
status                 0
subtotal               0
discount_amount        0
total                  0
placed_at              0
created_at             0
updated_at             0
dtype: int64

id duplicado: 0
order_number duplicado: 0


### D. Consulta ou código — SQL da Q1.1, no PostgreSQL

In [4]:
with open("../Submissão/Q1/1.1.sql") as f:
    query_q1 = f.read()

consultar(query_q1)

,total_linhas,total_colunas,data_minima,data_maxima,total_minimo,total_maximo,total_medio
0,48998,13,2020-01-01 01:19:28,2026-12-31 23:43:09,32.62,127262.02,28704.992077227642


### E. Resultado

48.998 linhas, 13 colunas, `created_at` de 2020-01-01 a 2026-12-31, `total` de R$ 32,62 a R$ 127.262,02, média R$ 28.704,99. Sem nulos nas colunas essenciais, sem `id`/`order_number` duplicado.

### F. Interpretação

**Q1.2 — valor médio de `total`:** R$ 28.704,99.

**Q1.3 — diagnóstico** (texto completo em `../Submissão/Q1/1.3.md`): os valores mais altos de `total` são plausíveis no contexto de varejo náutico (motores, embarcações), mas isso não é uma confirmação — só relacionando com `order_items` dá pra validar se são pedidos legítimos de grande porte. O único nulo relevante é `salesperson_id`, concentrado no canal `ecommerce`, mas sem ser uma regra fechada (30% dos pedidos `ecommerce` têm vendedor preenchido). A tabela está pronta para as agregações simples pedidas aqui; para as análises de negócio das próximas questões, ainda falta confirmar a consistência das demais tabelas e decidir o que fazer com pedidos `cancelled`/`draft`.

## Q2 — Schema

### A. Objetivo

Ler os 24 CSVs e gerar um `schema.sql` (DDL do PostgreSQL) — um `CREATE TABLE` por arquivo — usando só biblioteca padrão do Python.

### B. Caminho adotado

Por coluna: primeiro um override semântico explícito (lista curta, só para colunas que parecem número mas são identificador — CPF, chave de nota fiscal, código NCM, código de barras/EAN), senão uma varredura completa dos valores decidindo entre vazio, booleano, inteiro, decimal, data, timestamp ou texto. Script completo em `../Submissão/Q2/infer_schema.py`; já rodou via terminal antes deste notebook (ver `../README.md`).

### C. Ponto de atenção

Um campo só com dígitos nem sempre é número. `employees.cpf` e `products.ncm_code`, por exemplo, virariam `BIGINT` sem o override, porque a amostra atual não tem nenhum zero à esquerda — não é garantia, é coincidência dos dados de hoje. Zero à esquerda observado em qualquer linha também bloqueia tipo numérico automaticamente, mesmo sem override — é o caso do CEP em `addresses`/`locations`, que fica como texto sem precisar entrar na lista de exceções.

### D. Consulta ou código — conferindo o schema já criado

In [5]:
consultar(
    "SELECT COUNT(*) AS total_tabelas FROM information_schema.tables WHERE table_schema = current_schema()"
)

,total_tabelas
0,24


In [6]:
consultar(
    """
    SELECT table_name, column_name, data_type
    FROM information_schema.columns
    WHERE (table_name, column_name) IN (
        ('customers', 'tax_id'),
        ('employees', 'cpf'),
        ('products', 'ncm_code'),
        ('fiscal_invoices', 'nfe_access_key'),
        ('fiscal_invoices', 'series'),
        ('product_variants', 'barcode_ean'),
        ('product_variants', 'sale_price')
    )
    ORDER BY table_name, column_name
    """
)

,table_name,column_name,data_type
0,customers,tax_id,text
1,employees,cpf,text
2,fiscal_invoices,nfe_access_key,text
3,fiscal_invoices,series,text
4,product_variants,barcode_ean,text
5,product_variants,sale_price,numeric
6,products,ncm_code,text


### E. Resultado

24 tabelas. Os identificadores sensíveis (CPF, chave de NF-e, série, código de barras, código NCM) aparecem como `text`; `sale_price` aparece como `numeric` — nenhum vira número inteiro por engano.

### F. Interpretação

O resultado confirma que a regra de inferência funcionou nos casos mais sensíveis: colunas que são só dígitos, mas representam documento/código (não quantidade), ficaram como texto — preservando qualquer zero à esquerda que existisse. Colunas de preço/taxa ficaram numéricas, prontas para soma e média nas próximas questões.

## Q3 — Carregamento

### A. Objetivo

Carregar os 24 CSVs no schema da Q2, sem nenhum tratamento — sem remover nulo, sem corrigir caractere especial.

### B. Caminho adotado

`COPY FROM STDIN` (não `INSERT` nem `COPY` do lado do servidor), com as colunas declaradas na ordem do cabeçalho do CSV. As 24 tabelas carregam numa única transação: só confirma (`COMMIT`) depois de reconciliar a contagem de cada tabela contra o CSV de origem. Script completo em `../Submissão/Q3/load_data.py`; já rodou via terminal antes deste notebook.

### C. Ponto de atenção

O schema da Q2 não tem chave única (é uma camada bruta, de propósito) — rodar a carga duas vezes duplicaria os dados silenciosamente. Por isso o script recusa rodar se o destino já tiver alguma linha, em vez de duplicar sem avisar.

### D. Consulta ou código — conferindo a carga já feita

In [7]:
contagem_q3_2 = consultar(
    """
    SELECT 'customers' AS tabela, COUNT(*) AS linhas FROM customers
    UNION ALL SELECT 'orders', COUNT(*) FROM orders
    UNION ALL SELECT 'order_items', COUNT(*) FROM order_items
    UNION ALL SELECT 'payments', COUNT(*) FROM payments
    """
)
contagem_q3_2

,tabela,linhas
0,customers,2000
1,orders,48998
2,payments,53546
3,order_items,147320


In [8]:
print("Q3.2 (customers + orders + order_items + payments):", contagem_q3_2["linhas"].sum())

Q3.2 (customers + orders + order_items + payments): 251864


### E. Resultado

`customers` = 2.000, `orders` = 48.998, `order_items` = 147.320, `payments` = 53.546 — soma **251.864** (resposta da Q3.2, `../Submissão/Q3/3.2.md`). No total, as 24 tabelas somam 433.424 linhas, igual à soma dos 24 CSVs, sem divergência em nenhuma.

### F. Interpretação

A contagem bateu exatamente com o CSV de origem em todas as tabelas — é essa reconciliação, tabela por tabela (não só o total geral), que garante que a carga não perdeu nem duplicou nenhuma linha: uma falta numa tabela nunca fica escondida por um excesso em outra.

## Q4 — Análise de Clientes

### A. Objetivo

Identificar os 10 clientes com maior ticket médio entre os que compraram de 13 ou mais categorias diferentes, e descobrir qual categoria eles mais compraram em quantidade de itens.

### B. Caminho adotado

`orders` tem uma linha por pedido; `order_items` tem uma linha por item. Por isso faturamento, frequência e ticket médio vêm de uma consulta que só olha `orders`; a diversidade de categorias vem de uma consulta separada, na cadeia de itens (`orders → order_items → product_variants → products`) — as duas se juntam depois, por `customer_id`.

### C. Ponto de atenção

Se faturamento/frequência fossem calculados depois de um `JOIN` com `order_items` (necessário pra chegar em categoria), um pedido de 3 itens contaria 3 vezes — faturamento e frequência inflados.

### D. Consulta ou código

In [9]:
top_10 = consultar(
    """
    WITH metricas_pedidos AS (
        SELECT
            customer_id,
            SUM(total) AS faturamento_total,
            COUNT(id) AS frequencia,
            SUM(total) / COUNT(id) AS ticket_medio
        FROM orders
        GROUP BY customer_id
    ),
    diversidade_clientes AS (
        SELECT
            o.customer_id,
            COUNT(DISTINCT p.category_id) AS diversidade_categorias
        FROM orders o
        JOIN order_items oi ON oi.order_id = o.id
        JOIN product_variants pv ON pv.id = oi.product_variant_id
        JOIN products p ON p.id = pv.product_id
        GROUP BY o.customer_id
    )
    SELECT
        mp.customer_id,
        ROUND(mp.faturamento_total, 2) AS faturamento_total,
        mp.frequencia,
        ROUND(mp.ticket_medio, 2) AS ticket_medio,
        dc.diversidade_categorias
    FROM metricas_pedidos mp
    JOIN diversidade_clientes dc ON dc.customer_id = mp.customer_id
    WHERE dc.diversidade_categorias >= 13
    ORDER BY mp.ticket_medio DESC, mp.customer_id ASC
    LIMIT 10
    """
)

top_10

,customer_id,faturamento_total,frequencia,ticket_medio,diversidade_categorias
0,22,1087838.44,26,41839.94,14
1,1477,916262.58,22,41648.30,14
2,929,1082775.89,26,41645.23,14
3,1116,655737.20,16,40983.58,14
4,1691,815471.30,20,40773.57,14
5,774,726127.99,18,40340.44,14
6,1470,1040553.09,26,40021.27,14
7,1599,997616.46,25,39904.66,14
8,965,677297.78,17,39841.05,14
9,1722,1146455.22,29,39532.94,14


In [10]:
categoria_top10 = consultar(
    """
    WITH metricas_pedidos AS (
        SELECT customer_id, SUM(total) / COUNT(id) AS ticket_medio
        FROM orders
        GROUP BY customer_id
    ),
    diversidade_clientes AS (
        SELECT o.customer_id, COUNT(DISTINCT p.category_id) AS diversidade_categorias
        FROM orders o
        JOIN order_items oi ON oi.order_id = o.id
        JOIN product_variants pv ON pv.id = oi.product_variant_id
        JOIN products p ON p.id = pv.product_id
        GROUP BY o.customer_id
    ),
    top_10 AS (
        SELECT mp.customer_id
        FROM metricas_pedidos mp
        JOIN diversidade_clientes dc ON dc.customer_id = mp.customer_id
        WHERE dc.diversidade_categorias >= 13
        ORDER BY mp.ticket_medio DESC, mp.customer_id ASC
        LIMIT 10
    )
    SELECT
        c.id AS category_id,
        c.name AS categoria,
        SUM(oi.quantity) AS quantidade_total
    FROM top_10 t
    JOIN orders o ON o.customer_id = t.customer_id
    JOIN order_items oi ON oi.order_id = o.id
    JOIN product_variants pv ON pv.id = oi.product_variant_id
    JOIN products p ON p.id = pv.product_id
    JOIN categories c ON c.id = p.category_id
    GROUP BY c.id, c.name
    ORDER BY quantidade_total DESC, category_id ASC
    """
)

categoria_top10

,category_id,categoria,quantidade_total
0,8,Hélices,492
1,3,Coletes Salva-Vidas,393
2,5,Eletrônica Náutica,392
3,7,Âncoras,387
4,10,Iluminação,333
5,12,Manutenção,330
6,13,SEGURANÇA,325
7,6,Velas,313
8,11,Pintura Marítima,307
9,9,Acessórios de Convés,305


### E. Resultado

Top 10 clientes por ticket médio, todos com diversidade 14. Categoria líder entre esses 10: **Hélices** (`category_id = 8`), 492 itens — a segunda colocada (Coletes Salva-Vidas, 393) fica bem atrás.

### F. Interpretação

A restrição ao Top 10 vem da ordem das operações: os 10 `customer_id` são isolados numa CTE antes de qualquer coisa, e a segunda consulta parte dela — nenhum pedido de cliente fora do ranking entra na soma de quantidade. Detalhe completo em `../Submissão/Q4/4.2.md`.

## Q5 — Dimensão de Calendário

### A. Objetivo

O Sr. Almir quer saber qual dia da semana tem a pior média de vendas nas lojas físicas (`channel = 'pos'`), pra decidir se vale fechar a loja nesse dia.

### B. Caminho adotado

Um estagiário fictício agrupou direto nos pedidos físicos e achou Domingo ótimo — mas dias em que a loja abriu e vendeu zero não existem entre as vendas `pos` agregadas (ausência de linha, não linha com valor zero). A correção é uma dimensão de calendário, gerada com `generate_series` (uma linha por data entre o `MIN` e o `MAX` de `placed_at::date` observados no arquivo — não `CURRENT_DATE`, pra consulta ficar reproduzível).

### C. Ponto de atenção

O `LEFT JOIN` precisa partir do calendário, não das vendas — e `COALESCE(venda_diaria, 0)` precisa vir antes da média, porque `AVG()` ignora `NULL` mas não ignora zero. Sem isso, os dias sem venda ficariam fora do cálculo e o erro do estagiário se repetiria.

### D. Consulta ou código

In [11]:
calendario_semana = consultar(
    """
    WITH limites AS (
        SELECT MIN(placed_at::date) AS data_inicial, MAX(placed_at::date) AS data_final
        FROM orders
    ),
    datas AS (
        SELECT generate_series(data_inicial, data_final, INTERVAL '1 day')::date AS data
        FROM limites
    ),
    calendario AS (
        SELECT
            data,
            EXTRACT(ISODOW FROM data)::int AS numero_dia_semana,
            CASE EXTRACT(ISODOW FROM data)::int
                WHEN 1 THEN 'Segunda-feira' WHEN 2 THEN 'Terça-feira' WHEN 3 THEN 'Quarta-feira'
                WHEN 4 THEN 'Quinta-feira' WHEN 5 THEN 'Sexta-feira' WHEN 6 THEN 'Sábado'
                WHEN 7 THEN 'Domingo'
            END AS dia_semana
        FROM datas
    ),
    vendas_diarias AS (
        SELECT placed_at::date AS data, SUM(total) AS venda_diaria
        FROM orders
        WHERE channel = 'pos'
        GROUP BY placed_at::date
    ),
    calendario_com_vendas AS (
        SELECT c.data, c.numero_dia_semana, c.dia_semana, COALESCE(v.venda_diaria, 0) AS venda_diaria
        FROM calendario c
        LEFT JOIN vendas_diarias v ON v.data = c.data
    )
    SELECT
        numero_dia_semana,
        dia_semana,
        COUNT(*) AS dias_no_calendario,
        COUNT(*) FILTER (WHERE venda_diaria = 0) AS dias_sem_venda,
        ROUND(AVG(venda_diaria), 2) AS media_vendas_diarias
    FROM calendario_com_vendas
    GROUP BY numero_dia_semana, dia_semana
    ORDER BY media_vendas_diarias ASC, numero_dia_semana ASC
    """
)

calendario_semana

,numero_dia_semana,dia_semana,dias_no_calendario,dias_sem_venda,media_vendas_diarias
0,4,Quinta-feira,366,20,157154.32
1,7,Domingo,365,12,157616.13
2,1,Segunda-feira,365,7,158241.15
3,6,Sábado,365,11,164858.27
4,2,Terça-feira,365,8,166118.83
5,5,Sexta-feira,365,10,170193.68
6,3,Quarta-feira,366,10,173605.44


### E. Resultado

Pior dia: **Quinta-feira**, média de R$ 157.154,32 (20 das 366 quintas do período sem nenhuma venda física). Domingo — o dia que o estagiário achou ótimo — fica em segundo lugar entre os piores, não em primeiro entre os melhores.

### F. Interpretação

A diferença entre contar só os dias com venda e contar todos os dias do calendário é o que muda a resposta: sem os zeros, a Quinta-feira pareceria valer bem mais. O resultado mostra qual dia teve a pior média histórica segundo as premissas do desafio — não prova sozinho que fechar a loja nesse dia é a melhor decisão (isso exigiria considerar custo fixo, margem e sazonalidade). Detalhe completo em `../Submissão/Q5/5.2.md`.

In [12]:
pg.close()